# Assault DDQN - HU008 MLflow experiment tracking


## 1. Bootstrap Local -> GitHub -> Colab

In [ ]:
import os


os.environ.setdefault("ASSAULT_BOOTSTRAP_REF", "feature/hu008-mlflow-tracking")
os.environ.setdefault("ASSAULT_E2E_REQUIRE_CUDA", "1")


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/j-mauro-r/reinforcement_learning_reto_1.git"
COLAB_ROOT = Path("/content/reinforcement_learning_reto_1")
BOOTSTRAP_REF = os.environ.get("ASSAULT_BOOTSTRAP_REF", "main")
BOOTSTRAP_COMMIT = os.environ.get("ASSAULT_BOOTSTRAP_COMMIT") or None
INSTALL_DEPENDENCIES = os.environ.get("ASSAULT_INSTALL_DEPENDENCIES", "1") == "1"


def _running_in_colab():
    try:
        import google.colab  # type: ignore  # noqa: F401
    except ImportError:
        return False
    return True


def _git_output(args, cwd):
    return subprocess.check_output(["git", *args], cwd=str(cwd), text=True).strip()


if _running_in_colab():
    if not (COLAB_ROOT / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(COLAB_ROOT)], check=True)
    subprocess.run(["git", "fetch", "--prune", "origin"], cwd=str(COLAB_ROOT), check=True)
    provisional_ref = BOOTSTRAP_COMMIT or f"origin/{BOOTSTRAP_REF}"
    provisional_sha = _git_output(["rev-parse", "--verify", f"{provisional_ref}^{{commit}}"], COLAB_ROOT)
    subprocess.run(["git", "checkout", "--detach", provisional_sha], cwd=str(COLAB_ROOT), check=True)
    ASSAULT_DIR = COLAB_ROOT / "2_Assault"
else:
    PROJECT_ROOT = Path(_git_output(["rev-parse", "--show-toplevel"], Path.cwd()))
    ASSAULT_DIR = PROJECT_ROOT / "2_Assault"

for path in (ASSAULT_DIR, ASSAULT_DIR.parent):
    value = str(path.resolve())
    if value in sys.path:
        sys.path.remove(value)
    sys.path.insert(0, value)

from src.execution_bootstrap import (
    install_project_requirements,
    prepare_execution_environment,
    verify_environment_import,
)

bootstrap = prepare_execution_environment(
    requested_ref=BOOTSTRAP_REF,
    requested_commit=BOOTSTRAP_COMMIT,
    repo_url=REPO_URL,
    colab_root=COLAB_ROOT,
)

PROJECT_ROOT = bootstrap.repo_root
ASSAULT_DIR = bootstrap.assault_dir

if INSTALL_DEPENDENCIES:
    install_project_requirements(bootstrap.requirements_path)

environment_source = verify_environment_import(bootstrap)
bootstrap.as_dict()


In [ ]:
# Optional notebook diagnostics go after the project bootstrap/import cells.
# This placeholder intentionally avoids importing src before sys.path is configured.


## 2. Imports, configuration and tracking


In [ ]:
from pathlib import Path

from src.callbacks import TensorBoardLogger, load_tensorboard_scalars
from src.environment import create_assault_env, get_environment_metadata, validate_frameskip_once
from src.preflight import run_preflight_checks
from src.tracking import MLflowTracker
from src.training_session import run_training_session
from src.utils import get_runtime_info, load_yaml_config

config = load_yaml_config(ASSAULT_DIR / "configs" / "ddqn_config.yaml")
seed = int(config["reproducibility"]["seed"])
print("PROJECT_ROOT:", PROJECT_ROOT)
print("ASSAULT_DIR:", ASSAULT_DIR)
print("BOOTSTRAP_REF:", BOOTSTRAP_REF)
print("BOOTSTRAP_COMMIT:", BOOTSTRAP_COMMIT or "<none>")
print("EXECUTED_SHA:", bootstrap.resolved_sha)
print("src.environment:", environment_source)
config


## 3. Runtime and hardware

In [ ]:
runtime_info = get_runtime_info()
runtime_info


## 4. HU002 environment contract

In [ ]:
train_env = create_assault_env(config, mode="train", seed=seed)
eval_env = create_assault_env(config, mode="eval", seed=seed + 1)

obs, info = train_env.reset(seed=seed)
metadata = get_environment_metadata(train_env, config, mode="train", seed=seed)

print("Observation shape:", obs.shape)
print("Observation dtype:", obs.dtype)
print("Action space:", train_env.action_space)
print("Action meanings:", train_env.unwrapped.get_action_meanings())
print("Initial info:", info)
print("Metadata:", metadata)


## 5. HU002 autovalidations

In [ ]:
assert obs.shape == (4, 84, 84)
assert str(obs.dtype) == "uint8"
assert train_env.action_space.n == 7
assert train_env.observation_space.shape == eval_env.observation_space.shape
assert train_env.observation_space.dtype == eval_env.observation_space.dtype
assert validate_frameskip_once(train_env, expected_frameskip=4, steps=5)

obs, info = train_env.reset(seed=seed)
for step in range(100):
    action = int(train_env.action_space.sample())
    obs, reward, terminated, truncated, info = train_env.step(action)
    assert obs.shape == (4, 84, 84)
    assert str(obs.dtype) == "uint8"
    if terminated or truncated:
        obs, info = train_env.reset()

print("HU002 validations passed.")
train_env.close()
eval_env.close()


## 6. HU004 preflight gate

In [ ]:
preflight_report = run_preflight_checks(config)
print(preflight_report.format_summary())
preflight_report.as_dict()


## 7. Abort if preflight fails

In [ ]:
if not preflight_report.ready_for_training:
    raise RuntimeError("READY_FOR_TRAINING=False; HU005 training aborted.")
print("READY_FOR_TRAINING=True")


## 8. HU008 multi-session configuration


In [ ]:
def _env_flag(name: str, default: bool) -> bool:
    raw = os.environ.get(name)
    if raw is None:
        return default
    return raw.strip().lower() in {"1", "true", "yes", "y"}


def _optional_env(name: str, default=None):
    value = os.environ.get(name)
    if value is None or not value.strip():
        return default
    return value.strip()


e2e_config = config.get("e2e_smoke", {})
mlflow_config = config.get("mlflow", {})
RUN_ID = _optional_env("ASSAULT_RUN_ID", config["checkpointing"]["run_id"] + "_hu008_smoke")
CHECKPOINT_DIR = Path(_optional_env("ASSAULT_CHECKPOINT_DIR", str(ASSAULT_DIR / config["checkpointing"]["directory"])))
TENSORBOARD_DIR = Path(_optional_env("ASSAULT_TENSORBOARD_DIR", str(ASSAULT_DIR / config.get("tensorboard", {}).get("directory", "logs/tensorboard"))))
MLFLOW_TRACKING_MODE = _optional_env("ASSAULT_MLFLOW_TRACKING_MODE", mlflow_config.get("tracking_mode", "new")).lower()
MLFLOW_RUN_ID = _optional_env("ASSAULT_MLFLOW_RUN_ID", mlflow_config.get("mlflow_run_id"))
MLFLOW_SESSION_ID = _optional_env("ASSAULT_MLFLOW_SESSION_ID", mlflow_config.get("tracking_session_id"))
CHECKPOINT_INPUT_REFERENCE = _optional_env("ASSAULT_MLFLOW_CHECKPOINT_INPUT")
RESUME_MODE = _optional_env("ASSAULT_RESUME_MODE", "resume_full")
_default_target = e2e_config.get("final_timesteps") if MLFLOW_TRACKING_MODE == "resume" else e2e_config.get("segment_a_timesteps")
SESSION_TARGET_TIMESTEPS = int(_optional_env("ASSAULT_MLFLOW_SESSION_TARGET_TIMESTEPS", _default_target or config["training"]["total_timesteps"]))

if mlflow_config.get("enabled", False) and not MLFLOW_SESSION_ID:
    raise ValueError("ASSAULT_MLFLOW_SESSION_ID or mlflow.tracking_session_id is required when MLflow is enabled.")
if MLFLOW_TRACKING_MODE == "resume" and not MLFLOW_RUN_ID:
    raise ValueError("ASSAULT_MLFLOW_RUN_ID is required when ASSAULT_MLFLOW_TRACKING_MODE=resume.")
if MLFLOW_TRACKING_MODE == "resume" and not CHECKPOINT_INPUT_REFERENCE:
    raise ValueError("ASSAULT_MLFLOW_CHECKPOINT_INPUT is required when ASSAULT_MLFLOW_TRACKING_MODE=resume.")
if MLFLOW_TRACKING_MODE == "new" and CHECKPOINT_INPUT_REFERENCE:
    raise ValueError("ASSAULT_MLFLOW_CHECKPOINT_INPUT must be empty when ASSAULT_MLFLOW_TRACKING_MODE=new.")

print("HU008 project_run_id:", RUN_ID)
print("Checkpoint directory:", CHECKPOINT_DIR)
print("TensorBoard directory:", TENSORBOARD_DIR)
print("Session target timesteps:", SESSION_TARGET_TIMESTEPS)
print("Resume mode:", RESUME_MODE)
print("MLflow enabled:", mlflow_config.get("enabled"))
print("MLflow tracking mode:", MLFLOW_TRACKING_MODE)
print("MLflow configured URI:", os.environ.get("ASSAULT_MLFLOW_TRACKING_URI") or mlflow_config.get("tracking_uri") or mlflow_config.get("local_directory"))
print("MLflow experiment:", os.environ.get("ASSAULT_MLFLOW_EXPERIMENT") or mlflow_config.get("experiment_name"))
print("MLflow run id:", MLFLOW_RUN_ID or "<new run>")
print("MLflow tracking session id:", MLFLOW_SESSION_ID)
print("Checkpoint input reference:", CHECKPOINT_INPUT_REFERENCE or "<none>")


## 9. HU008 tracked training session


In [ ]:
import copy

MLFLOW_TRACKING_PASS = False
MLFLOW_SESSION_ARTIFACTS = []
CHECKPOINT_INPUT_LOADED = False
RESTORED_GLOBAL_STEP = None
REPLAY_BUFFER_RESTORED = False
MULTISESSION_CHECKPOINT_RESUME_PASS = False
LOCAL_E2E_SMOKE_PASS = False
E2E_SMOKE_PASS = False

session_config = copy.deepcopy(config)
session_config.setdefault("training", {})["total_timesteps"] = SESSION_TARGET_TIMESTEPS
mlflow_tracker = MLflowTracker.from_config(session_config)
mlflow_metadata = mlflow_tracker.start_run(
    project_run_id=RUN_ID,
    tracking_mode=MLFLOW_TRACKING_MODE,
    mlflow_run_id=MLFLOW_RUN_ID,
    run_name=RUN_ID,
    tags={"stage": "HU008"},
    tracking_session_id=MLFLOW_SESSION_ID,
)
print("MLflow tracking URI:", mlflow_metadata.tracking_uri)
print("MLflow experiment name:", mlflow_metadata.experiment_name)
print("project_run_id:", mlflow_metadata.project_run_id)
print("mlflow_run_id:", mlflow_metadata.mlflow_run_id)
print("tracking_session_id:", mlflow_metadata.tracking_session_id)

try:
    selected_runtime = "Google Colab" if _running_in_colab() else "local"
    selected_device = "cuda" if __import__("torch").cuda.is_available() else "cpu"
    mlflow_tracker.log_run_context(
        config=session_config,
        runtime_info=runtime_info,
        git_commit=bootstrap.resolved_sha,
        git_ref=BOOTSTRAP_REF,
        project_run_id=RUN_ID,
        action_space=str(metadata.action_space),
        observation_dtype=str(obs.dtype),
        runtime=selected_runtime,
        device=selected_device,
    )
    mlflow_tracker.log_config_snapshot(session_config)
    mlflow_tracker.log_runtime_metadata(
        runtime_info=runtime_info,
        git_commit=bootstrap.resolved_sha,
        runtime=selected_runtime,
        tracking_session_id=MLFLOW_SESSION_ID,
    )

    session_summary = run_training_session(
        config=session_config,
        checkpoint_root=CHECKPOINT_DIR,
        tensorboard_root=TENSORBOARD_DIR,
        run_id=RUN_ID,
        repo_path=PROJECT_ROOT,
        tracking_mode=MLFLOW_TRACKING_MODE,
        checkpoint_input=CHECKPOINT_INPUT_REFERENCE,
        resume_mode=RESUME_MODE,
        total_timesteps=SESSION_TARGET_TIMESTEPS,
        device=selected_device,
    )
    session_result = session_summary.as_dict()
    checkpoint_output_reference = session_summary.checkpoint_output_reference
    session_initial_step = int(session_summary.initial_global_step)
    session_final_step = int(session_summary.final_global_step)
    CHECKPOINT_INPUT_LOADED = bool(session_summary.checkpoint_input_loaded)
    RESTORED_GLOBAL_STEP = session_summary.restored_global_step
    REPLAY_BUFFER_RESTORED = bool(session_summary.replay_buffer_restored)
    MULTISESSION_CHECKPOINT_RESUME_PASS = bool(
        MLFLOW_TRACKING_MODE == "resume"
        and CHECKPOINT_INPUT_LOADED
        and RESTORED_GLOBAL_STEP == session_initial_step
        and REPLAY_BUFFER_RESTORED
        and session_final_step > session_initial_step
        and session_summary.checkpoint_input_reference == CHECKPOINT_INPUT_REFERENCE
    )

    mlflow_tracker.log_training_summary(session_summary.training, tracking_session_id=MLFLOW_SESSION_ID)
    mlflow_tracker.log_checkpoint_reference(
        session_summary.checkpoint,
        resume_mode=RESUME_MODE if MLFLOW_TRACKING_MODE == "resume" else "new",
        project_run_id=RUN_ID,
        checkpoint_input_reference=CHECKPOINT_INPUT_REFERENCE,
        checkpoint_output_reference=checkpoint_output_reference,
        tracking_session_id=MLFLOW_SESSION_ID,
    )
    mlflow_tracker.log_dict_artifact(session_result, "training_session_summary.json", tracking_session_id=MLFLOW_SESSION_ID)
    mlflow_tracker.log_session_metadata(
        tracking_mode=MLFLOW_TRACKING_MODE,
        runtime_info=runtime_info,
        git_commit=bootstrap.resolved_sha,
        git_ref=BOOTSTRAP_REF,
        runtime=selected_runtime,
        device=selected_device,
        initial_global_step=session_initial_step,
        final_global_step=session_final_step,
        checkpoint_input_reference=CHECKPOINT_INPUT_REFERENCE,
        checkpoint_output_reference=checkpoint_output_reference,
        checkpoint_input_loaded=CHECKPOINT_INPUT_LOADED,
        restored_checkpoint_path=session_summary.restored_checkpoint_path,
        restored_global_step=RESTORED_GLOBAL_STEP,
        replay_buffer_restored=REPLAY_BUFFER_RESTORED,
        resume_mode=session_summary.resume_mode,
        tracking_session_id=MLFLOW_SESSION_ID,
    )
    MLFLOW_SESSION_ARTIFACTS = [artifact.path for artifact in mlflow_tracker.list_session_artifacts(MLFLOW_SESSION_ID)]
    queried_run = mlflow_tracker.get_run(mlflow_metadata.mlflow_run_id) if mlflow_metadata.enabled else None
    required_session_artifacts = {
        f"sessions/{MLFLOW_SESSION_ID}/session_metadata.json",
        f"sessions/{MLFLOW_SESSION_ID}/runtime.json",
        f"sessions/{MLFLOW_SESSION_ID}/training_summary.json",
        f"sessions/{MLFLOW_SESSION_ID}/checkpoint_reference.json",
        f"sessions/{MLFLOW_SESSION_ID}/training_session_summary.json",
    }
    MLFLOW_TRACKING_PASS = bool(
        not mlflow_metadata.enabled
        or (
            queried_run is not None
            and queried_run.info.run_id == mlflow_metadata.mlflow_run_id
            and queried_run.data.params.get("identity.project_run_id") == RUN_ID
            and queried_run.data.tags.get("latest_tracking_session_id") == MLFLOW_SESSION_ID
            and "train/final_global_step" in queried_run.data.metrics
            and required_session_artifacts.issubset(set(MLFLOW_SESSION_ARTIFACTS))
            and (MLFLOW_TRACKING_MODE != "resume" or MULTISESSION_CHECKPOINT_RESUME_PASS)
        )
    )
finally:
    mlflow_tracker.end_run(status="FINISHED" if MLFLOW_TRACKING_PASS else "FAILED")

session_result


## 10. HU008 result gates


In [ ]:
assert session_summary.initial_global_step == session_initial_step
assert session_summary.final_global_step == session_final_step
assert session_final_step == SESSION_TARGET_TIMESTEPS
assert session_summary.checkpoint.path.exists()
if MLFLOW_TRACKING_MODE == "new":
    assert session_initial_step == 0
    assert not CHECKPOINT_INPUT_LOADED
    assert CHECKPOINT_INPUT_REFERENCE is None
elif MLFLOW_TRACKING_MODE == "resume":
    assert CHECKPOINT_INPUT_LOADED
    assert RESTORED_GLOBAL_STEP == session_initial_step
    assert REPLAY_BUFFER_RESTORED
    assert session_initial_step > 0
    assert session_final_step > session_initial_step
    assert session_summary.checkpoint_input_reference == CHECKPOINT_INPUT_REFERENCE
    assert MULTISESSION_CHECKPOINT_RESUME_PASS
else:
    raise AssertionError(f"Unsupported MLflow tracking mode: {MLFLOW_TRACKING_MODE}")
if mlflow_config.get("enabled", False):
    assert MLFLOW_TRACKING_PASS, "MLflow tracking validation did not pass."

print("HU008 MLflow tracking status")
print("runtime:", selected_runtime)
print("device:", selected_device)
print("tracking_mode:", MLFLOW_TRACKING_MODE)
print("project_run_id:", mlflow_metadata.project_run_id)
print("mlflow_run_id:", mlflow_metadata.mlflow_run_id)
print("tracking_session_id:", mlflow_metadata.tracking_session_id)
print("checkpoint_input_reference:", CHECKPOINT_INPUT_REFERENCE)
print("checkpoint_input_loaded:", CHECKPOINT_INPUT_LOADED)
print("restored_global_step:", RESTORED_GLOBAL_STEP)
print("replay_buffer_restored:", REPLAY_BUFFER_RESTORED)
print("initial_global_step:", session_initial_step)
print("final_global_step:", session_final_step)
print("checkpoint_output_reference:", checkpoint_output_reference)
print("mlflow_tracking_uri:", mlflow_metadata.tracking_uri)
print("mlflow_experiment:", mlflow_metadata.experiment_name)
print("observation:", metadata.observation_shape, metadata.observation_dtype)
print("action_space:", metadata.action_space)
print("Preflight READY_FOR_TRAINING:", preflight_report.ready_for_training)
print("training_updates:", session_summary.training.updates_count)
print("training_initial_step:", session_summary.training.initial_global_step)
print("training_final_step:", session_summary.training.global_step)
print("checkpoint_path:", session_summary.checkpoint.path)
print("checkpoint_size_bytes:", session_summary.checkpoint.size_bytes)
print("session_artifacts:", MLFLOW_SESSION_ARTIFACTS)
print("MULTISESSION_CHECKPOINT_RESUME_PASS=", MULTISESSION_CHECKPOINT_RESUME_PASS)
print("LOCAL_E2E_SMOKE_PASS=", LOCAL_E2E_SMOKE_PASS)
print("E2E_SMOKE_PASS=", E2E_SMOKE_PASS)
print("MLFLOW_TRACKING_PASS=", MLFLOW_TRACKING_PASS)
